## 1. ONNX Export 환경 및 학습 모델 준비

1. 필수 라이브러리 불러오기

In [1]:
%pip install -q onnx onnxruntime

In [2]:
%pip install -q --upgrade onnx onnxscript onnxruntime

In [3]:
%pip install -q \
    "numpy==2.2.6" \
    "numba==0.61.2" \
    "anomalib==2.6.2" \
    jedi

In [4]:
%pip install --force-reinstall --no-cache-dir "numpy==2.1.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 144.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


In [5]:
from pathlib import Path
import importlib.util

import torch

from anomalib.data import MVTecAD
from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.deploy import ExportType

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


2. ONNX 관련 패키지 설치 여부 확인

In [6]:
packages = [
    "onnx",
    "onnxruntime",
]

for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package:12s}: {installed}")

onnx        : True
onnxruntime : True


3. 현재 실행 환경 확인

In [7]:
import anomalib

print("PyTorch :", torch.__version__)
print("Anomalib:", anomalib.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

PyTorch : 2.11.0+cu128
Anomalib: 2.6.2
CUDA    : True
GPU     : Tesla T4


4. 저장된 Checkpoint 확인

In [8]:
checkpoint_path = Path(
    "./models/patchcore_bottle.ckpt"
)

print("Checkpoint exists:", checkpoint_path.exists())
print("Checkpoint path  :", checkpoint_path)

Checkpoint exists: True
Checkpoint path  : models/patchcore_bottle.ckpt


5. MVTec AD DataModule 구성

In [9]:
datamodule = MVTecAD(
    root="./datasets/MVTecAD",
    category="bottle",
    train_batch_size=32,
    eval_batch_size=32,
)

datamodule.prepare_data()
datamodule.setup()

6. 학습과 동일한 PatchCore 구조 생성

In [10]:
pre_processor = Patchcore.configure_pre_processor(
    image_size=(256, 256),
    center_crop_size=(256, 256),
)

model = Patchcore(
    backbone="wide_resnet50_2",
    layers=["layer2", "layer3"],
    pre_trained=True,
    coreset_sampling_ratio=0.1,
    num_neighbors=9,
    pre_processor=pre_processor,
)

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


7. Export용 Engine 생성

In [11]:
engine = Engine(
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)

## 2. PatchCore 배포 구조 분리 및 Feature Extractor Export

1. 학습된 PatchCore Checkpoint 불러오기

In [12]:
trained_model = Patchcore.load_from_checkpoint(
    str(checkpoint_path),
    weights_only=False,
)

trained_model.eval()

print("Checkpoint loaded.")
print(
    "Memory bank shape:",
    trained_model.model.memory_bank.shape,
)

Checkpoint loaded.
Memory bank shape: torch.Size([21401, 1536])


2. Memory Bank 별도 저장

In [13]:
import numpy as np

memory_bank = (
    trained_model.model
    .memory_bank
    .detach()
    .cpu()
    .numpy()
)

memory_bank_path = Path(
    "./models/patchcore_memory_bank.npy"
)

np.save(
    memory_bank_path,
    memory_bank,
)

print("Memory bank shape:", memory_bank.shape)
print("Saved to         :", memory_bank_path)

Memory bank shape: (21401, 1536)
Saved to         : models/patchcore_memory_bank.npy


3. Feature Extractor Wrapper 정의

In [14]:
import torch
import torch.nn.functional as F


class PatchcoreFeatureExtractor(torch.nn.Module):
    def __init__(self, patchcore_model):
        super().__init__()

        self.feature_extractor = (
            patchcore_model.feature_extractor
        )

        self.feature_pooler = (
            patchcore_model.feature_pooler
        )

        self.layers = list(
            patchcore_model.layers
        )

    def forward(self, x):
        features = self.feature_extractor(x)

        features = {
            layer: self.feature_pooler(
                features[layer]
            )
            for layer in self.layers
        }

        embedding = features[
            self.layers[0]
        ]

        for layer in self.layers[1:]:
            layer_embedding = features[layer]

            layer_embedding = F.interpolate(
                layer_embedding,
                size=embedding.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            embedding = torch.cat(
                (
                    embedding,
                    layer_embedding,
                ),
                dim=1,
            )

        return embedding

4. Feature Extractor 모델 생성

In [15]:
feature_model = PatchcoreFeatureExtractor(
    trained_model.model
)

feature_model.eval()

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

feature_model = feature_model.to(device)

print("Device:", device)

Device: cuda


5. Dummy Input 생성

In [16]:
dummy_input = torch.randn(
    1,
    3,
    256,
    256,
    device=device,
)

print("Input shape:", dummy_input.shape)

Input shape: torch.Size([1, 3, 256, 256])


6. PyTorch Feature Embedding 출력 확인

In [17]:
with torch.no_grad():
    embedding = feature_model(
        dummy_input
    )

print(
    "Embedding shape:",
    embedding.shape,
)

Embedding shape: torch.Size([1, 1536, 32, 32])


## 3. PatchCore Feature Extractor ONNX Export

1. ONNX 저장 폴더 생성

In [18]:
onnx_export_dir = Path(
    "./models/onnx"
)

onnx_export_dir.mkdir(
    parents=True,
    exist_ok=True,
)

feature_onnx_path = (
    onnx_export_dir
    / "patchcore_feature_extractor.onnx"
)

print(
    "ONNX output:",
    feature_onnx_path,
)

ONNX output: models/onnx/patchcore_feature_extractor.onnx


2. Feature Extractor ONNX Export

In [19]:
torch.onnx.export(
    feature_model,
    (dummy_input,),
    str(feature_onnx_path),

    input_names=[
        "input"
    ],

    output_names=[
        "embedding"
    ],

    dynamo=True,
)

print(
    "ONNX export completed:",
    feature_onnx_path,
)

[torch.onnx] Obtain model graph for `PatchcoreFeatureExtractor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PatchcoreFeatureExtractor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX export completed: models/onnx/patchcore_feature_extractor.onnx


3. ONNX 파일 생성 확인

In [20]:
print(
    "Exists:",
    feature_onnx_path.exists(),
)

if feature_onnx_path.exists():
    size_mb = (
        feature_onnx_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"ONNX size: {size_mb:.2f} MB"
    )

Exists: True
ONNX size: 0.19 MB


## 4. ONNX 모델 구조 검증

1. ONNX 모델 Load

In [21]:
import onnx

onnx_model = onnx.load(
    str(feature_onnx_path)
)

print(
    "ONNX model loaded."
)

ONNX model loaded.


2. ONNX Checker 실행

In [22]:
onnx.checker.check_model(
    onnx_model
)

print(
    "ONNX model validation passed."
)

ONNX model validation passed.


3. ONNX Input 확인

In [23]:
print("Inputs:")

for model_input in onnx_model.graph.input:
    print(
        "-",
        model_input.name,
    )

Inputs:
- input


4. ONNX Output 확인

In [24]:
print("Outputs:")

for model_output in onnx_model.graph.output:
    print(
        "-",
        model_output.name,
    )

Outputs:
- embedding


5. ONNX Operator 확인

In [25]:
from collections import Counter

op_counts = Counter(
    node.op_type
    for node in onnx_model.graph.node
)

for op_name, count in op_counts.most_common():
    print(
        f"{op_name:20s}: {count}"
    )

Conv                : 43
Relu                : 40
Add                 : 13
AveragePool         : 2
MaxPool             : 1
Resize              : 1
Concat              : 1


## 2. PatchCore 배포 구조 분리 및 Feature Extractor 준비

1. 학습된 PatchCore checkpoint 불러오기

In [27]:
trained_model = Patchcore.load_from_checkpoint(
    str(checkpoint_path),
    weights_only=False,
)

trained_model.eval()

print("Checkpoint loaded.")
print(
    "Memory bank shape:",
    trained_model.model.memory_bank.shape,
)

Checkpoint loaded.
Memory bank shape: torch.Size([21401, 1536])


2. Memory Bank 별도 저장

In [28]:
import numpy as np

memory_bank = (
    trained_model.model
    .memory_bank
    .detach()
    .cpu()
    .numpy()
)

memory_bank_path = Path(
    "./models/patchcore_memory_bank.npy"
)

np.save(
    memory_bank_path,
    memory_bank,
)

print("Memory bank shape:", memory_bank.shape)
print("Saved to         :", memory_bank_path)

Memory bank shape: (21401, 1536)
Saved to         : models/patchcore_memory_bank.npy


3. Feature Extractor Wrapper 정의

In [29]:
import torch
import torch.nn.functional as F


class PatchcoreFeatureExtractor(torch.nn.Module):
    def __init__(self, patchcore_model):
        super().__init__()

        self.feature_extractor = patchcore_model.feature_extractor
        self.feature_pooler = patchcore_model.feature_pooler
        self.layers = list(patchcore_model.layers)

    def forward(self, x):
        features = self.feature_extractor(x)

        features = {
            layer: self.feature_pooler(features[layer])
            for layer in self.layers
        }

        embedding = features[self.layers[0]]

        for layer in self.layers[1:]:
            layer_embedding = features[layer]

            layer_embedding = F.interpolate(
                layer_embedding,
                size=embedding.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            embedding = torch.cat(
                (embedding, layer_embedding),
                dim=1,
            )

        return embedding

4. Feature Extractor 모델 생성

In [30]:
feature_model = PatchcoreFeatureExtractor(
    trained_model.model
)

feature_model.eval()

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

feature_model = feature_model.to(device)

print("Device:", device)

Device: cuda


5. Dummy Input 생성

In [31]:
dummy_input = torch.randn(
    1,
    3,
    256,
    256,
    device=device,
)

print("Input shape:", dummy_input.shape)

Input shape: torch.Size([1, 3, 256, 256])


6. PyTorch Feature Embedding 확인

In [32]:
with torch.no_grad():
    embedding = feature_model(
        dummy_input
    )

print("Embedding shape:", embedding.shape)

Embedding shape: torch.Size([1, 1536, 32, 32])


## 3. PatchCore Feature Extractor ONNX Export

1. ONNX 저장 경로 생성

In [33]:
onnx_export_dir = Path(
    "./models/onnx"
)

onnx_export_dir.mkdir(
    parents=True,
    exist_ok=True,
)

feature_onnx_path = (
    onnx_export_dir
    / "patchcore_feature_extractor.onnx"
)

print("ONNX output:", feature_onnx_path)

ONNX output: models/onnx/patchcore_feature_extractor.onnx


2. Feature Extractor ONNX Export

In [34]:
torch.onnx.export(
    feature_model,
    (dummy_input,),
    str(feature_onnx_path),

    input_names=[
        "input"
    ],

    output_names=[
        "embedding"
    ],

    dynamo=True,
)

print(
    "ONNX export completed:",
    feature_onnx_path,
)

[torch.onnx] Obtain model graph for `PatchcoreFeatureExtractor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PatchcoreFeatureExtractor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX export completed: models/onnx/patchcore_feature_extractor.onnx


3. ONNX 파일 생성 확인

In [36]:
print(
    "Exists:",
    feature_onnx_path.exists(),
)

if feature_onnx_path.exists():
    size_mb = (
        feature_onnx_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"ONNX size: {size_mb:.2f} MB"
    )

Exists: True
ONNX size: 0.19 MB


## 4. ONNX 모델 구조 검증

1. ONNX Load

In [37]:
import onnx

onnx_model = onnx.load(
    str(feature_onnx_path)
)

print("ONNX model loaded.")

ONNX model loaded.


2. ONNX Checker

In [38]:
onnx.checker.check_model(
    onnx_model
)

print(
    "ONNX model validation passed."
)

ONNX model validation passed.


3. Input / Output 확인

In [39]:
print("Inputs:")

for model_input in onnx_model.graph.input:
    print("-", model_input.name)

print("\nOutputs:")

for model_output in onnx_model.graph.output:
    print("-", model_output.name)

Inputs:
- input

Outputs:
- embedding


4. ONNX 연산 확인

In [40]:
from collections import Counter

op_counts = Counter(
    node.op_type
    for node in onnx_model.graph.node
)

for op_name, count in op_counts.most_common():
    print(
        f"{op_name:20s}: {count}"
    )

Conv                : 43
Relu                : 40
Add                 : 13
AveragePool         : 2
MaxPool             : 1
Resize              : 1
Concat              : 1


## 3. PatchCore Feature Extractor ONNX Export

1. ONNX 저장 경로 생성

In [41]:
onnx_export_dir = Path("./models/onnx")

onnx_export_dir.mkdir(
    parents=True,
    exist_ok=True,
)

feature_onnx_path = (
    onnx_export_dir
    / "patchcore_feature_extractor.onnx"
)

print("ONNX output:", feature_onnx_path)

ONNX output: models/onnx/patchcore_feature_extractor.onnx


2. Feature Extractor ONNX Export

In [42]:
torch.onnx.export(
    feature_model,
    (dummy_input,),
    str(feature_onnx_path),
    input_names=["input"],
    output_names=["embedding"],
    dynamo=True,
)

print(
    "ONNX export completed:",
    feature_onnx_path,
)

[torch.onnx] Obtain model graph for `PatchcoreFeatureExtractor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PatchcoreFeatureExtractor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX export completed: models/onnx/patchcore_feature_extractor.onnx


3. ONNX 파일 생성 확인

In [43]:
print(
    "Exists:",
    feature_onnx_path.exists(),
)

if feature_onnx_path.exists():
    size_mb = (
        feature_onnx_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"ONNX size: {size_mb:.2f} MB"
    )

Exists: True
ONNX size: 0.19 MB


## 4. ONNX 모델 구조 및 출력 정합성 검증

1. ONNX 모델 Load 및 구조 검증

In [44]:
!ls -lh ./models/onnx

total 96M
-rw-r--r-- 1 root root 193K Sep 16 02:41 patchcore_feature_extractor.onnx
-rw-r--r-- 1 root root  95M Sep 16 02:41 patchcore_feature_extractor.onnx.data


2. ONNX 모델 Load

In [45]:
import onnx

onnx_model = onnx.load(
    str(feature_onnx_path)
)

print("ONNX model loaded.")

ONNX model loaded.


3. ONNX 구조 검증

In [46]:
onnx.checker.check_model(
    onnx_model
)

print("ONNX model validation passed.")

ONNX model validation passed.


In [47]:
def get_tensor_shape(value_info):
    return [
        dim.dim_value
        if dim.dim_value > 0
        else dim.dim_param
        for dim in value_info.type.tensor_type.shape.dim
    ]


for model_input in onnx_model.graph.input:
    print(
        "Input :",
        model_input.name,
        get_tensor_shape(model_input),
    )

for model_output in onnx_model.graph.output:
    print(
        "Output:",
        model_output.name,
        get_tensor_shape(model_output),
    )

Input : input [1, 3, 256, 256]
Output: embedding [1, 1536, 32, 32]


In [48]:
from collections import Counter

op_counts = Counter(
    node.op_type
    for node in onnx_model.graph.node
)

for op_name, count in op_counts.most_common():
    print(
        f"{op_name:20s}: {count}"
    )

Conv                : 43
Relu                : 40
Add                 : 13
AveragePool         : 2
MaxPool             : 1
Resize              : 1
Concat              : 1


In [49]:
import onnxruntime as ort

ort_session = ort.InferenceSession(
    str(feature_onnx_path),
    providers=[
        "CPUExecutionProvider"
    ],
)

print(
    "Providers:",
    ort_session.get_providers(),
)

Providers: ['CPUExecutionProvider']


In [50]:
ort_input = ort_session.get_inputs()[0]
ort_output = ort_session.get_outputs()[0]

print(
    "Input :",
    ort_input.name,
    ort_input.shape,
)

print(
    "Output:",
    ort_output.name,
    ort_output.shape,
)

Input : input [1, 3, 256, 256]
Output: embedding [1, 1536, 32, 32]


In [51]:
dummy_numpy = (
    dummy_input
    .detach()
    .cpu()
    .numpy()
)

onnx_outputs = ort_session.run(
    ["embedding"],
    {
        "input": dummy_numpy
    },
)

onnx_embedding = onnx_outputs[0]

print(
    "ONNX embedding shape:",
    onnx_embedding.shape,
)

ONNX embedding shape: (1, 1536, 32, 32)


In [52]:
pytorch_embedding = (
    embedding
    .detach()
    .cpu()
    .numpy()
)

print(
    "PyTorch embedding shape:",
    pytorch_embedding.shape,
)

PyTorch embedding shape: (1, 1536, 32, 32)


In [53]:
import numpy as np

absolute_error = np.abs(
    pytorch_embedding
    - onnx_embedding
)

print(
    "Max absolute error :",
    absolute_error.max(),
)

print(
    "Mean absolute error:",
    absolute_error.mean(),
)

Max absolute error : 1.5258789e-05
Mean absolute error: 2.7548654e-07


In [54]:
is_close = np.allclose(
    pytorch_embedding,
    onnx_embedding,
    rtol=1e-3,
    atol=1e-4,
)

print(
    "PyTorch ≈ ONNX:",
    is_close,
)

PyTorch ≈ ONNX: True
